# 🐄 Cattle Breed Classifier — Colab Runner

This notebook sets up the environment on **Google Colab** and runs all training notebooks sequentially.

### What this does
1. Verifies GPU availability
2. Clones the repo (dataset included)
3. Installs missing dependencies
4. Runs each notebook in order:
   - `00_data_audit` → `01_mlp_baseline` → `02_cnn_from_scratch` → `03_resnet_transfer_learning` → `04_vit_transfer_learning` → `05_model_comparison`

> ⚠️ **Runtime**: Select **GPU** runtime before running: *Runtime → Change runtime type → T4 GPU*

## 0. Check GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'Memory:          {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## 1. Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/ajitsingh98/cattle-breed-classifier-webapp.git'
REPO_DIR = '/content/cattle-breed-classifier-webapp'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'\nWorking directory: {os.getcwd()}')

## 2. Install Dependencies

In [ ]:
# Colab already has PyTorch, torchvision, numpy, pandas, matplotlib, seaborn, PIL, scikit-learn.
# Install only the missing packages.
!pip install -q PyYAML gdown aiofiles aiohttp nest_asyncio

## 3. Set Up Python Path

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(REPO_DIR).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Create artifact directories
for d in ['ml/artifacts/manifests', 'ml/artifacts/checkpoints',
          'ml/artifacts/figures', 'ml/artifacts/logs', 'ml/artifacts/reports']:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# Verify dataset exists
data_dir = PROJECT_ROOT / 'Cattle_Resized'
class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
total_images = sum(len(list(d.glob('*'))) for d in class_dirs)
print(f'Dataset: {len(class_dirs)} breeds, {total_images} images')
print(f'Project root: {PROJECT_ROOT}')

## 4. Run All Notebooks

Each notebook is executed in order using `nbconvert`. Output is displayed inline.

You can also **skip this cell** and open each notebook individually from the file browser on the left:
> `cattle-breed-classifier-webapp/ml/notebooks/`

In [ ]:
!pip install -q papermill jupyter ipykernel

In [ ]:
import subprocess, time

NOTEBOOKS = [
    '00_data_audit.ipynb',
    '01_mlp_baseline.ipynb',
    '02_cnn_from_scratch.ipynb',
    '03_resnet_transfer_learning.ipynb',
    '04_vit_transfer_learning.ipynb',
    '05_model_comparison.ipynb',
]

NOTEBOOK_DIR = PROJECT_ROOT / 'ml' / 'notebooks'
results = {}

for nb_name in NOTEBOOKS:
    nb_path = NOTEBOOK_DIR / nb_name
    print(f'\n{"=" * 60}')
    print(f'▶ Running: {nb_name}')
    print(f'{"=" * 60}')

    start = time.time()
    # We use papermill because it supports streaming cell outputs (--log-output)
    result = subprocess.run(
        [
            "papermill",
            str(nb_path),
            str(nb_path), # overwrite inplace so output is saved in the notebook
            "--log-output",
            "--kernel", "python3"
        ],
        cwd=str(PROJECT_ROOT),
        env={**os.environ, "PYTHONPATH": str(PROJECT_ROOT)},
    )
    elapsed = time.time() - start

    if result.returncode == 0:
        status = '✅ PASSED'
    else:
        status = '❌ FAILED'

    results[nb_name] = {'status': status, 'time': elapsed}
    print(f'{status} ({elapsed:.1f}s)')

# Summary
print(f'\n{"=" * 60}')
print('Summary')
print(f'{"=" * 60}')
for nb, info in results.items():
    print(f"  {info['status']}  {nb:45s}  {info['time']:6.1f}s")
total_time = sum(r['time'] for r in results.values())
print(f'\nTotal time: {total_time/60:.1f} minutes')

## 5. Download Artifacts (Optional)

After training completes, download the best model checkpoint and reports.

In [ ]:
# List generated artifacts
import glob

print('=== Checkpoints ===')
for f in sorted(glob.glob(str(PROJECT_ROOT / 'ml/artifacts/checkpoints/*.pth'))):
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {Path(f).name:40s} {size_mb:8.1f} MB')

print('\n=== Figures ===')
for f in sorted(glob.glob(str(PROJECT_ROOT / 'ml/artifacts/figures/*.png'))):
    print(f'  {Path(f).name}')

print('\n=== Reports ===')
for f in sorted(glob.glob(str(PROJECT_ROOT / 'ml/artifacts/reports/*.json'))):
    print(f'  {Path(f).name}')

In [ ]:
# Zip and download artifacts
!cd {REPO_DIR} && zip -r /content/artifacts.zip ml/artifacts/

from google.colab import files
files.download('/content/artifacts.zip')
print('\n✅ Download started! Check your browser downloads.')

---

### 💡 Running Notebooks Individually

Instead of the automated runner above, you can open each notebook directly:

1. In the **Colab file browser** (left panel), navigate to:  
   `cattle-breed-classifier-webapp/ml/notebooks/`
2. Double-click any `.ipynb` file to open it in a new tab
3. Run cells with `Shift+Enter`

**Important**: Each notebook auto-detects the project root, so they work both from the automated runner and when opened individually.